In [ ]:
%load_ext cudf.pandas

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%%RecordEvent
# %load_ext cudf.pandas
import sys, os
tpch_parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, tpch_parent)
import pandas as pd
# from tpch import load_part, load_lineitem, load_supplier, load_orders, load_customer, load_nation, load_region, q08
DATA_ROOT = "/home/dias-benchmarks/tpch/data"
STORAGE_OPTS = {}




In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q08/q08_rewrite/checkpoints/pre_cell_0.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 0 ###

def load_part(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/part.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
part = load_part(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q08/q08_rewrite/checkpoints/pre_cell_1.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 1 ###

def load_lineitem(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/lineitem.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    print(df.columns)
    df["L_SHIPDATE"] = pd.to_datetime(df.L_SHIPDATE, format="%Y-%m-%d")
    df["L_RECEIPTDATE"] = pd.to_datetime(df.L_RECEIPTDATE, format="%Y-%m-%d")
    df["L_COMMITDATE"] = pd.to_datetime(df.L_COMMITDATE, format="%Y-%m-%d")
    return df
lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)


In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q08/q08_rewrite/checkpoints/pre_cell_2.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 2 ###

def load_supplier(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/supplier.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
supplier = load_supplier(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q08/q08_rewrite/checkpoints/pre_cell_3.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 3 ###

def load_orders(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/orders.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    df["O_ORDERDATE"] = pd.to_datetime(df.O_ORDERDATE, format="%Y-%m-%d")
    return df
orders = load_orders(DATA_ROOT, **STORAGE_OPTS)



In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q08/q08_rewrite/checkpoints/pre_cell_4.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 4 ###


def load_customer(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/customer.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
customer = load_customer(DATA_ROOT, **STORAGE_OPTS)


In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q08/q08_rewrite/checkpoints/pre_cell_5.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 5 ###

def load_nation(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/nation.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
nation = load_nation(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q08/q08_rewrite/checkpoints/pre_cell_6.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 6 ###

def load_region(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/region.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
region = load_region(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q08/q08_rewrite/checkpoints/pre_cell_7.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 7 ###


part_filtered = part[(part["P_TYPE"] == "ECONOMY ANODIZED STEEL")]
part_filtered = part_filtered.loc[:, ["P_PARTKEY"]]
lineitem_filtered = lineitem.loc[:, ["L_PARTKEY", "L_SUPPKEY", "L_ORDERKEY"]]
lineitem_filtered["VOLUME"] = lineitem["L_EXTENDEDPRICE"] * (
    1.0 - lineitem["L_DISCOUNT"]
)
total = part_filtered.merge(
    lineitem_filtered, left_on="P_PARTKEY", right_on="L_PARTKEY", how="inner"
)
total = total.loc[:, ["L_SUPPKEY", "L_ORDERKEY", "VOLUME"]]
supplier_filtered = supplier.loc[:, ["S_SUPPKEY", "S_NATIONKEY"]]
total = total.merge(
    supplier_filtered, left_on="L_SUPPKEY", right_on="S_SUPPKEY", how="inner"
)
total = total.loc[:, ["L_ORDERKEY", "VOLUME", "S_NATIONKEY"]]
orders_filtered = orders[
    (orders["O_ORDERDATE"] >= pd.Timestamp("1995-01-01"))
    & (orders["O_ORDERDATE"] < pd.Timestamp("1997-01-01"))
]
orders_filtered["O_YEAR"] = orders_filtered["O_ORDERDATE"].dt.year
orders_filtered = orders_filtered.loc[:, ["O_ORDERKEY", "O_CUSTKEY", "O_YEAR"]]
total = total.merge(
    orders_filtered, left_on="L_ORDERKEY", right_on="O_ORDERKEY", how="inner"
)
total = total.loc[:, ["VOLUME", "S_NATIONKEY", "O_CUSTKEY", "O_YEAR"]]
customer_filtered = customer.loc[:, ["C_CUSTKEY", "C_NATIONKEY"]]
total = total.merge(
    customer_filtered, left_on="O_CUSTKEY", right_on="C_CUSTKEY", how="inner"
)
total = total.loc[:, ["VOLUME", "S_NATIONKEY", "O_YEAR", "C_NATIONKEY"]]
n1_filtered = nation.loc[:, ["N_NATIONKEY", "N_REGIONKEY"]]
n2_filtered = nation.loc[:, ["N_NATIONKEY", "N_NAME"]].rename(
    columns={"N_NAME": "NATION"}
)
total = total.merge(
    n1_filtered, left_on="C_NATIONKEY", right_on="N_NATIONKEY", how="inner"
)
total = total.loc[:, ["VOLUME", "S_NATIONKEY", "O_YEAR", "N_REGIONKEY"]]
total = total.merge(
    n2_filtered, left_on="S_NATIONKEY", right_on="N_NATIONKEY", how="inner"
)
total = total.loc[:, ["VOLUME", "O_YEAR", "N_REGIONKEY", "NATION"]]
region_filtered = region[(region["R_NAME"] == "AMERICA")]
region_filtered = region_filtered.loc[:, ["R_REGIONKEY"]]
total = total.merge(
    region_filtered, left_on="N_REGIONKEY", right_on="R_REGIONKEY", how="inner"
)
total = total.loc[:, ["VOLUME", "O_YEAR", "NATION"]]

def udf(df):
    demonimator = df["VOLUME"].sum()
    df = df[df["NATION"] == "BRAZIL"]
    numerator = df["VOLUME"].sum()
    return numerator / demonimator

total = total.groupby("O_YEAR", as_index=False).apply(udf)
total.columns = ["O_YEAR", "MKT_SHARE"]
total = total.sort_values(by=["O_YEAR"], ascending=[True])



In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q08/q08_rewrite/checkpoints/pre_cell_8.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 8 ###

total.info()